In [1]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import yfinance as yf
import ta
import math
from fredapi import Fred
import logging
import ecbdata
import requests
import json

Fred_api = "e886df7269c2c4e6209754d4ea0371d5"
Grok_api = "xai-zBPdvxZOTUXY6Uaos1erYo0RIQultvO4E1TRvZ4eGhkNpV66ZYjH8Uke9Flf5iUp7xqg5SttSJyp9ccr"
ALPHA_VANTAGE_API_KEY = "PFG4O7CL1VMB51GE"
NEWS_API_KEY = "32a3030e3c804c1da8e950585321222c"

xai_base_url = "https://api.x.ai/v1"
xai_headers = {
    "Authorization": f"Bearer {Grok_api}",
    "Content-Type": "application/json"
}

def query_grok(prompt, data=None):
    content = prompt + (f"\nData: {json.dumps(data, default=str)}" if data else "")
    payload = {
        "model": "grok-2-1212",
        "messages": [
            {"role": "user", "content": content}
        ],
        "max_tokens": 200
    }
    try:
        response = requests.post(f"{xai_base_url}/chat/completions", headers=xai_headers, json=payload)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"].strip()
    except Exception as e:
        print(f"Grok API error: {e}")
        return None

In [2]:
test_response = query_grok("Say 'hello' if you’re working.")
print(test_response)

Hello! I'm here and ready to assist you.


# List Of Tradeable Pairs And Indicators

In [3]:
# Initialize MetaTrader 5 connection
mt5.initialize()

# Updated list of currency pairs
pairs = [
    "EURUSD",  # Euro / US Dollar
    "EURCHF",  # Euro / Swiss Franc
    "EURJPY",  # Euro / Japanese Yen
    "USDCHF",  # US Dollar / Swiss Franc
    "CHFJPY",  # Swiss Franc / Japanese Yen
    
    "USDJPY"   # US Dollar / Japanese Yen
]

currencies = [
   "DX-Y.NYB", # Dollar Currency Index
    "^XDE",    # Euro Currency Index
    "^XDS",    # Chf Currency Index
    "^XDN"     # Yen Currency Index
]

# Function to get the latest Ask and Bid prices for a given pair
def get_latest_prices(symbol):
    # Get the latest tick data for the symbol
    tick = mt5.symbol_info_tick(symbol)
    if tick is None:
        print(f"Failed to get latest tick data for {symbol}")
        return None, None
    return tick.ask, tick.bid

# Function to get historical data for a given pair
def get_historical_data(symbol, timeframe=mt5.TIMEFRAME_M15, n_bars=64):
    # Fetch historical data
    rates = mt5.copy_rates_from_pos(symbol, timeframe, 0, n_bars)
    if rates is None or len(rates) == 0:
        print(f"Failed to get historical data for {symbol}")
        return None
    data = pd.DataFrame(rates)
    data['time'] = pd.to_datetime(data['time'], unit='s')
    return data

# Function to calculate EMA, RSI, and ATR for a given pair
def calculate_indicators(symbol):
    # Get historical data for the pair
    data = get_historical_data(symbol)
    if data is None:
        return None, None, None, None

    # Calculate EMA 64
    data['EMA_64'] = ta.trend.ema_indicator(data['close'], window=64)

    # Calculate RSI 16
    data['RSI_16'] = ta.momentum.rsi(data['close'], window=16)

    # Calculate ATR 16
    data['ATR_16'] = ta.volatility.average_true_range(data['high'], data['low'], data['close'], window=16)

    # Get the latest values of the indicators
    latest_price = data['close'].iloc[-1]
    latest_ema = data['EMA_64'].iloc[-1]
    latest_rsi = data['RSI_16'].iloc[-1]
    latest_atr = data['ATR_16'].iloc[-1]

    return latest_price, latest_ema, latest_rsi, latest_atr

# Macroeconomic Data

In [4]:
import pandas as pd
from fredapi import Fred
import logging
from ecbdata import ecbdata

# Initialize FRED with your API key
fred = Fred(api_key=Fred_api)  # Replace with your actual API key

# Define the countries and their respective indicators
countries = {
    'USA': {'gdp': 'GDPC1', 'unemp': 'UNRATE', 'interest': 'FEDFUNDS', 'inflation': 'CPIAUCSL'},
    'Europe': {'gdp': 'CLVMEURSCAB1GQEA19', 'unemp': 'LRUNTTTTQZA156S', 'interest': 'IR3TIB01EZQ156N', 'inflation': 'CPHPTT01EZQ659N'},
    'Switzerland': {'gdp': 'CLVMNACSAB1GQCH', 'unemp': 'LRUNTTTTCHQ156S', 'interest': 'IR3TIB01CHQ156N', 'inflation': 'CHECPIALLMINMEI'},
    'Japan': {'gdp': 'JPNRGDPEXP', 'unemp': 'LRUNTTTTJPQ156S', 'interest': 'IR3TIB01JPQ156N', 'inflation': 'JPNCPIALLMINMEI'}
}


# Initialize empty dictionaries for storing the economic data
growth_rates = {}
unemp_rates = {}
interest_rates = {}
inflation_rates = {}
fallbacks = {
    'Europe': {
        'inflation': 'ICP.M.U2.N.000000.4.ANR',  # Alternative Europe inflation series
        'unemp': 'LFSI.M.U2.N.UNEHRT.TOTAL0.15_74.T',  # Alternative Europe unemployment series
    }
}

def get_10_years_Edata(symbol, country, indicator):
    try:
        economic_data = fred.get_series(symbol)
        economic_data = economic_data[economic_data.index >= pd.Timestamp.now() - pd.DateOffset(years=10)]
        if economic_data.empty:
            raise ValueError(f"No data returned for {symbol}")
        logging.info(f"Successfully fetched {indicator} data for {country} using {symbol} ({len(economic_data)} points)")
        return economic_data
    except Exception as e:
        logging.error(f"Error fetching {indicator} data for {country}: {e}")
        return None
def calculate_growth_rate(gdp_series):
    """Calculate annualized quarterly GDP growth rate"""
    if gdp_series is None or len(gdp_series) < 2:
        return None
    gdp_now = gdp_series.iloc[-1]
    gdp_previous = gdp_series.iloc[-2]
    return ((gdp_now - gdp_previous) / gdp_previous) * 100 * 4

def get_latest_value(series):
    """Get the most recent value from a series"""
    if series is None or len(series) < 1:
        return None
    return series.iloc[-1]

def calculate_inflation_rate(cpi_series):
    """Calculate year-over-year inflation rate"""
    if cpi_series is None or len(cpi_series) < 13:  # Need 12 months + 1 for monthly data
        logging.warning(f"Insufficient data for inflation calculation: {len(cpi_series)} points")
        return None
    cpi_now = cpi_series.iloc[-1]
    cpi_year_ago = cpi_series.iloc[-13]  # Assumes monthly data
    return ((cpi_now - cpi_year_ago) / cpi_year_ago) * 100

# Update the economic data and store it in a DataFrame
def update_economic_data():
    global growth_rates, unemp_rates, interest_rates, inflation_rates
    for country, symbols in countries.items():
        # GDP Growth
        gdp_data = get_10_years_Edata(symbols['gdp'], country, 'gdp')
        if gdp_data is not None:
            growth_rates[country] = calculate_growth_rate(gdp_data)
        
        # Unemployment
        unemp_data = get_10_years_Edata(symbols['unemp'], country, 'unemployment')
        if unemp_data is not None:
            unemp_rates[country] = get_latest_value(unemp_data)
            
        
        # Interest Rates
        interest_data = get_10_years_Edata(symbols['interest'], country, 'interest')
        if interest_data is not None:
            interest_rates[country] = get_latest_value(interest_data)
        
        # Inflation
        inflation_data = get_10_years_Edata(symbols['inflation'], country, 'inflation')
        if inflation_data is not None:
            if country == 'Europe' and symbols['inflation'] == 'CPHPTT01EZQ659N':
                inflation_rates[country] = get_eur_inflation()
            else:
                inflation_rates[country] = calculate_inflation_rate(inflation_data)
          
def get_eur_inflation():
    economic_data = ecbdata.get_series('ICP.M.U2.N.000000.4.ANR', start='2024-01')
    latest = economic_data['OBS_VALUE'].iloc[-1]  # Fixed: Use 'economic_data'
    return latest

def get_eur_unemployment():
    economic_data = ecbdata.get_series('LFSI.M.U2.N.UNEHRT.TOTAL0.15_74.T', start='2024-01')
    latest = economic_data['OBS_VALUE'].iloc[-1]  # Fixed: Use 'economic_data'
    return latest

# Create a DataFrame to hold the economic data
def create_economic_dataframe():
    update_economic_data()
    
    # Combine the data into a DataFrame
    data = {
        'GDP Growth': growth_rates,
        'Unemployment': unemp_rates,
        'Interest Rate': interest_rates,
        'Inflation Rate': inflation_rates
    }
    
    df = pd.DataFrame(data)
    df.loc['Japan', "Inflation Rate"] = 4
    df.loc['Europe', 'Unemployment'] = get_eur_unemployment() 
    df.loc['Europe', 'Inflation Rate'] =get_eur_inflation()
    return df
df = create_economic_dataframe()

df


ERROR:root:Error fetching unemployment data for Europe: Bad Request.  The series does not exist.
ERROR:root:Error fetching inflation data for Europe: No data returned for CPHPTT01EZQ659N
ERROR:root:Error fetching unemployment data for Japan: None


,GDP Growth,Unemployment,Interest Rate,Inflation Rate
USA,2.324740,4.000000,4.330000,2.999413
Europe,0.203240,6.268223,2.996487,2.500000
Switzerland,1.536775,4.497941,0.767250,0.401528
Japan,1.233826,NaN,0.334667,4.000000


# Currency Index Data

In [5]:
def get_currency_data(currency):
    # Fetch DXY historical data (last 10 days, 15m interval)
    dxy = yf.Ticker(currency)
    data = dxy.history(period="10d", interval="15m")
    
    # Recalculate RSI & EMA
    data["RSI_16"] = ta.momentum.rsi(data["Close"], window=64)
    data["EMA_64"] = ta.trend.ema_indicator(data["Close"], window=196)

    latest_price = data['Close'].iloc[-1]
    latest_ema = data['EMA_64'].iloc[-1]
    latest_rsi = data['RSI_16'].iloc[-1]
    return latest_price, latest_ema, latest_rsi

def get_currencies_table(currencies):
    dict = {}
    for i in currencies:
        name = i
        if i == "DX-Y.NYB":
            name = "USD"
        elif i == "^XDE" :
            name = "EUR"
        elif i== "^XDS" :
            name = "CHF"
        elif i== "^XDN" :
            name = "JPY"
        
        latest_price, latest_ema, latest_rsi = get_currency_data(i)
        dict.update({name: [latest_price, latest_ema, latest_rsi ]})
    data = pd.DataFrame.from_dict(dict, orient='index', columns=['Price', 'EMA', 'RSI'])
    return data

get_currencies_table(currencies)

,Price,EMA,RSI
USD,107.181000,106.577243,62.074513
EUR,104.078003,104.680773,40.026827
CHF,111.209999,111.377883,45.254603
JPY,66.703102,66.618715,47.898272


# Pip Value

In [6]:
def get_pip_value(symbol):
    dec = 0.0001
    if "JPY" in symbol:
        dec = 0.01
    latest_price, latest_ema, latest_rsi, latest_atr = calculate_indicators(symbol)
    pip_value = (dec * 100000) / latest_price
    return pip_value

# Position Size

In [7]:
def get_position_size(pair, stop_loss):
    account_info = mt5.account_info()
    balance = account_info.balance
    risk_amount = 0.005 * balance
    size = risk_amount / ( stop_loss * get_pip_value(pair))
    size = round(size, 2)
    return size

# Bias

In [8]:
# from imfdatapy.imf import IMF
# growth_rates, unemp_rates, interest_rates, inflation_rates

def update_debt_to_gdp():
    # """
    # Fetch debt-to-GDP ratios for 2025 (latest projection) and 2024 from IMF WEO for EU, US, Japan, and Switzerland.
    # Returns a dictionary with 'latest' (2025) and 'previous' (2024) sub-dictionaries for comparison.
    # """
    # try:
    #     imf = IMF()
    #     # Fetch "General government gross debt" (% of GDP) for EA (Euro Area), US, JP, CH
    #     debt_data = imf.get_series("WEO", "GGXWDG_NGDP", country=["EA", "US", "JP", "CH"], period="A")
        
    #     # Explicitly target 2025 (latest projection) and 2024 (previous year)
    #     debt_to_gdp = {
    #         "latest": {
    #             "EU": debt_data["EA"].loc[2025]["value"],
    #             "US": debt_data["US"].loc[2025]["value"],
    #             "Japan": debt_data["JP"].loc[2025]["value"],
    #             "Switzerland": debt_data["CH"].loc[2025]["value"]
    #         },
    #         "previous": {
    #             "EU": debt_data["EA"].loc[2024]["value"],
    #             "US": debt_data["US"].loc[2024]["value"],
    #             "Japan": debt_data["JP"].loc[2024]["value"],
    #             "Switzerland": debt_data["CH"].loc[2024]["value"]
    #         }
    #     }
    #     print(f"Debt-to-GDP fetched: Latest (2025 projection) vs Previous (2024)")
    #     return debt_to_gdp
    # except Exception as e:
    #     print(f"Error fetching debt-to-GDP data: {e}")
    #     # Fallback values for 2025 (projections) and 2024 based on trends from prior data
    return {
        "latest": {"EU": 89.0, "US": 122.0, "Japan": 261.0, "Switzerland": 43.5},  # 2025 projections
        "previous": {"EU": 88.5, "US": 120.0, "Japan": 260.0, "Switzerland": 43.3}  # 2024 estimates
    }

# Initialize debt_to_gdp dynamically
debt_to_gdp = update_debt_to_gdp()

def compare_economies(country1, country2):
    """
    Compare the economic data between two countries using latest data and year-over-year debt trends.
    Returns a 'buy' signal for country1 or 'sell' for country2 based on macroeconomic performance.
    Uses 5 factors, with debt-to-GDP trend as a tiebreaker, and a 2.5 threshold to minimize neutral outcomes.
    """
    buy_factors = 0
    sell_factors = 0
    
    # Compare GDP Growth Rates (weight: 1.5)
    if growth_rates.get(country1, 0) > growth_rates.get(country2, 0):
        buy_factors += 1.5
    elif growth_rates.get(country1, 0) < growth_rates.get(country2, 0):
        sell_factors += 1.5
    
    # Compare Unemployment Rates (lower is better, weight: 1)
    if unemp_rates.get(country1, float('inf')) < unemp_rates.get(country2, float('inf')):
        buy_factors += 1
    elif unemp_rates.get(country1, float('inf')) > unemp_rates.get(country2, float('inf')):
        sell_factors += 1
    
    # Compare Interest Rates (higher is generally better, weight: 1)
    if interest_rates.get(country1, 0) > interest_rates.get(country2, 0):
        buy_factors += 1
    elif interest_rates.get(country1, 0) < interest_rates.get(country2, 0):
        sell_factors += 1
    
    # Compare Inflation Rates (lower is better, weight: 1.5)
    if inflation_rates.get(country1, float('inf')) < inflation_rates.get(country2, float('inf')):
        buy_factors += 1.5
    elif inflation_rates.get(country1, float('inf')) > inflation_rates.get(country2, float('inf')):
        sell_factors += 1.5
    
    # Compare Debt-to-GDP Trend (lower latest vs previous is better, weight: 0.5)
    debt_trend1 = debt_to_gdp["latest"].get(country1, float('inf')) - debt_to_gdp["previous"].get(country1, float('inf'))
    debt_trend2 = debt_to_gdp["latest"].get(country2, float('inf')) - debt_to_gdp["previous"].get(country2, float('inf'))
    if debt_trend1 < debt_trend2:  # Smaller increase or larger decrease favors country1
        buy_factors += 0.5
    elif debt_trend1 > debt_trend2:
        sell_factors += 0.5
    
    # Determine the final bias with a 2.5 threshold to minimize neutral outcomes
    if buy_factors > 2.5:
        return 'buy'
    elif sell_factors > 2.5:
        return 'sell'
    else:
        return 'neutral'

def bias_for_pairs(pairs):
    bias_results = {}
    
    # Update debt_to_gdp before running comparisons
    global debt_to_gdp
    debt_to_gdp = update_debt_to_gdp()
    
    # Loop through each pair and get the macroeconomic comparison
    for i in pairs:
        if "EUR" in i and "USD" in i:
            bias = compare_economies("EU", "US")  # Compare Eurozone with USA
        elif "EUR" in i and "CHF" in i:
            bias = compare_economies("EU", "Switzerland")  # Compare Eurozone with Switzerland
        elif "EUR" in i and "JPY" in i:
            bias = compare_economies("EU", "Japan")  # Compare Eurozone with Japan
        elif "USD" in i and "CHF" in i:
            bias = compare_economies("US", "Switzerland")  # Compare USA with Switzerland
        elif "CHF" in i and "JPY" in i:
            bias = compare_economies("Switzerland", "Japan")  # Compare Switzerland with Japan
        elif "USD" in i and "JPY" in i:
            bias = compare_economies("US", "Japan")  # Compare USA with Japan
        
        # Update the dictionary with the bias result
        bias_results[i] = bias
    
    return bias_results


# Sentiment 

In [9]:
def get_sentiment(pair):
    table = get_currencies_table(currencies)
    signal1 = "neutral"
    signal2 = "neutral"
    sentiment = "neutral"
    for index in table.index:
        if pair.startswith(index):
            currency1 = index
        elif pair.endswith(index):
            currency2 = index
    if table.loc[currency1, "Price"] > table.loc[currency1, "EMA"] and table.loc[currency1, "RSI"] > 50:
        signal1 = "buy"
    elif table.loc[currency1, "Price"] < table.loc[currency1, "EMA"] and table.loc[currency1, "RSI"] < 50:
        signal1 = "sell"
    if table.loc[currency2, "Price"] > table.loc[currency2, "EMA"] and table.loc[currency2, "RSI"] > 50:
        signal2 = "buy"
    elif table.loc[currency2, "Price"] < table.loc[currency2, "EMA"] and table.loc[currency2, "RSI"] < 50:
        signal2 = "sell"
    if signal1 == "buy" and signal2 == "sell":
        sentiment = "buy"
    elif signal1 == "sell" and signal2 == "buy":
        sentiment = "sell"
    return sentiment

for i in pairs:
    print(i, get_sentiment(i))
    print()

EURUSD sell

EURCHF neutral

EURJPY neutral

USDCHF buy

CHFJPY neutral

USDJPY neutral



# Signals

In [10]:
def detect_rsi_divergence(symbol, timeframe=mt5.TIMEFRAME_M15, n_bars=64):
    data = get_historical_data(symbol, timeframe, n_bars)
    if data is None or len(data) < 20:
        return "neutral"

    data['RSI_16'] = ta.momentum.rsi(data['close'], window=16)
    data['price_high'] = data['close'][(data['close'] > data['close'].shift(1)) & 
                                       (data['close'] > data['close'].shift(-1)) & 
                                       (data['close'] > data['close'].shift(2)) & 
                                       (data['close'] > data['close'].shift(-2))]
    data['price_low'] = data['close'][(data['close'] < data['close'].shift(1)) & 
                                      (data['close'] < data['close'].shift(-1)) & 
                                      (data['close'] < data['close'].shift(2)) & 
                                      (data['close'] < data['close'].shift(-2))]
    data['rsi_high'] = data['RSI_16'][(data['RSI_16'] > data['RSI_16'].shift(1)) & 
                                      (data['RSI_16'] > data['RSI_16'].shift(-1)) & 
                                      (data['RSI_16'] > data['RSI_16'].shift(2)) & 
                                      (data['RSI_16'] > data['RSI_16'].shift(-2))]
    data['rsi_low'] = data['RSI_16'][(data['RSI_16'] < data['RSI_16'].shift(1)) & 
                                     (data['RSI_16'] < data['RSI_16'].shift(-1)) & 
                                     (data['RSI_16'] < data['RSI_16'].shift(2)) & 
                                     (data['RSI_16'] < data['RSI_16'].shift(-2))]

    price_highs = data['price_high'].dropna().tail(2)
    price_lows = data['price_low'].dropna().tail(2)
    rsi_highs = data['rsi_high'].dropna().tail(2)
    rsi_lows = data['rsi_low'].dropna().tail(2)

    if len(price_highs) < 2 or len(price_lows) < 2 or len(rsi_highs) < 2 or len(rsi_lows) < 2:
        return "neutral"

    ph1, ph2 = price_highs.iloc[0], price_highs.iloc[1]
    pl1, pl2 = price_lows.iloc[0], price_lows.iloc[1]
    rh1, rh2 = rsi_highs.iloc[0], rsi_highs.iloc[1]
    rl1, rl2 = rsi_lows.iloc[0], rsi_lows.iloc[1]

    if pl2 < pl1 and rl2 > rl1 and rl2 < 40:
        return "buy"
    elif ph2 > ph1 and rh2 < rh1 and rh2 > 60:
        return "sell"
    if pl2 > pl1 and rl2 < rl1 and rl2 < 50:
        return "buy"
    elif ph2 < ph1 and rh2 > rh1 and rh2 > 50:
        return "sell"
    return "neutral"

def detect_ict_signal(symbol, timeframe=mt5.TIMEFRAME_M15, n_bars=64):
    data = get_historical_data(symbol, timeframe, n_bars)
    if data is None or len(data) < 20:
        return "neutral"

    def is_buy_order_block(i):
        if data['close'][i] > data['open'][i] * 1.02:
            for j in range(1, 3):
                if data['low'][i+j] < data['low'][i]:
                    return False
            return True
        return False

    def is_sell_order_block(i):
        if data['close'][i] < data['open'][i] * 0.98:
            for j in range(1, 3):
                if data['high'][i+j] > data['high'][i]:
                    return False
            return True
        return False

    buy_blocks = [data['close'][i] for i in range(len(data)-3) if is_buy_order_block(i)]
    sell_blocks = [data['close'][i] for i in range(len(data)-3) if is_sell_order_block(i)]

    current_price = data['close'][0]
    pip_value = get_pip_value(symbol)
    threshold_pips = 10

    for block in buy_blocks:
        if abs(current_price - block) < threshold_pips * pip_value:
            return "buy"
    for block in sell_blocks:
        if abs(current_price - block) < threshold_pips * pip_value:
            return "sell"
    return "neutral"

def get_combined_signal(symbol):
    """Preview combined ICT and RSI signal direction and strength."""
    ict_signal = detect_ict_signal(symbol)
    rsi_signal = detect_rsi_divergence(symbol)
    
    # Map to scores for preview
    ict_score = 1 if ict_signal == "buy" else -1 if ict_signal == "sell" else 0
    rsi_score = 1 if rsi_signal == "buy" else -1 if rsi_signal == "sell" else 0
    
    # Calculate signal strength
    if ict_score == rsi_score and ict_score != 0:
        signal_strength = 1.0  # Full strength when aligned
        direction = "buy" if ict_score > 0 else "sell"
    elif (ict_score != 0 and rsi_score == 0) or (rsi_score != 0 and ict_score == 0):
        signal_strength = 0.5  # Half strength if one signal
        direction = "buy" if ict_score > 0 or rsi_score > 0 else "sell"
    else:
        signal_strength = 0.0  # Opposing or both neutral
        direction = "neutral"
    
    # Return combined signal with strength indicator
    if signal_strength == 0.0:
        return "neutral"
    return f"{direction} ({signal_strength:.1f})"


# News Reading

In [11]:
import requests
import json
from datetime import datetime, timedelta
import time

NEWS_API_KEY = "32a3030e3c804c1da8e950585321222c"
xai_base_url = "https://api.x.ai/v1"
Grok_api = "xai-zBPdvxZOTUXY6Uaos1erYo0RIQultvO4E1TRvZ4eGhkNpV66ZYjH8Uke9Flf5iUp7xqg5SttSJyp9ccr"
xai_headers = {
    "Authorization": f"Bearer {Grok_api}",
    "Content-Type": "application/json"
}

def get_news_score(pair):
    """Fetch a week's worth of news for the pair from NewsAPI and get a score from Grok."""
    query = pair
    url = f"https://newsapi.org/v2/everything?q={query}&from={(datetime.now() - timedelta(days=7)).strftime('%Y-%m-%d')}&sortBy=publishedAt&apiKey={NEWS_API_KEY}"
    
    try:
        print(f"{pair} - Fetching news from NewsAPI (last 7 days)")
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        news_data = response.json()
        
        if "articles" not in news_data or not news_data["articles"]:
            print(f"{pair} - No news found for the week, using static fallback")
            return 0
        
        articles = news_data["articles"][:5]
        news_text = "\n".join([f"Title: {a['title']}\nDesc: {a.get('description', '')}\n" for a in articles])
        print(f"{pair} - News fetched: {len(articles)} articles, sample: {news_text[:100]}...")
        
        prompt = (
            f"Analyze the following news articles like a financial analyst for {pair} over the past week and return a single numerical score "
            "between -1 and 1 (e.g., '0.5' or '-0.3'), where -1 is a strong sell, 0 is neutral, and 1 is a strong buy, "
            "based on their cumulative sentiment and likely price impact. Return only the number, no text, no explanation.\n"
            f"{news_text}"
        )
        payload = {
            "model": "grok-2-1212",
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 10
        }
        
        for attempt in range(3):
            try:
                print(f"{pair} - Attempt {attempt + 1}: Sending to Grok")
                response = requests.post(f"{xai_base_url}/chat/completions", headers=xai_headers, json=payload, timeout=5)
                response.raise_for_status()
                grok_response = response.json()["choices"][0]["message"]["content"].strip()
                print(f"{pair} - Grok response: {grok_response}")
                score = float(grok_response)
                if -1 <= score <= 1:
                    return score
                print(f"{pair} - Grok invalid news score: {score}, retrying...")
            
            except (requests.RequestException, ValueError, KeyError) as e:
                print(f"{pair} - Grok attempt {attempt + 1} failed: {e}")
                if attempt < 2:
                    time.sleep(1)
                continue
        
        print(f"{pair} - All Grok attempts failed, using static fallback")
        return 0
    
    except (requests.RequestException, ValueError) as e:
        print(f"{pair} - NewsAPI failed: {e}, using static fallback")
        return 0

# Test code
pairs = ["EURUSD", "EURCHF", "EURJPY", "USDCHF", "CHFJPY", "USDJPY"]
for pair in pairs:
    score = get_news_score(pair)
    print(f"{pair} - Weekly News Score: {score}\n")

EURUSD - Fetching news from NewsAPI (last 7 days)
EURUSD - News fetched: 5 articles, sample: Title: 【今週後半】に発表の注目イベントを厳選して『羊飼い秘蔵データ』を公開！◆2月24日～の週号◆
Desc: 【今週の後半】に発表予定の以下の注目イベントについて、
『羊飼い秘蔵のデータ』を...
EURUSD - Attempt 1: Sending to Grok
EURUSD - Grok response: -0.2
EURUSD - Weekly News Score: -0.2

EURCHF - Fetching news from NewsAPI (last 7 days)
EURCHF - No news found for the week, using static fallback
EURCHF - Weekly News Score: 0

EURJPY - Fetching news from NewsAPI (last 7 days)
EURJPY - No news found for the week, using static fallback
EURJPY - Weekly News Score: 0

USDCHF - Fetching news from NewsAPI (last 7 days)
USDCHF - No news found for the week, using static fallback
USDCHF - Weekly News Score: 0

CHFJPY - Fetching news from NewsAPI (last 7 days)
CHFJPY - No news found for the week, using static fallback
CHFJPY - Weekly News Score: 0

USDJPY - Fetching news from NewsAPI (last 7 days)
USDJPY - News fetched: 5 articles, sample: Title: 【今週後半】に発表の注目イベントを厳選して『羊飼い秘蔵データ』を公開！◆2月24日～の週

# Forex Pairs Data table

In [12]:
def get_data_table(pairs):
    dict = {}
    bias_results = bias_for_pairs(pairs)
    for i in pairs:
        ask, bid = get_latest_prices(i)
        latest_price, latest_ema, latest_rsi, latest_atr = calculate_indicators(i)
        dict.update({i: [ask,bid,latest_price, latest_ema, latest_rsi, latest_atr]})
    # Convert to DataFrame
    data = pd.DataFrame.from_dict(dict, orient='index', columns=['Ask', 'Bid', 'price', 'EMA', 'RSI', 'ATR'])
    
    data['Stop Loss']= data['ATR'] * 4
    data['Take Profit'] = data['Stop Loss'] * 2
    for i in pairs:
        dec = 10000
        if "JPY" in i:
            dec = 100
        stop_loss = round(data.loc[i, "Stop Loss"] * dec)
        take_profit = round(data.loc[i, "Take Profit"] * dec)
        data.loc[i,'Round Stop Loss'] = int(stop_loss)
        data.loc[i,'Round Take Profit'] = int(take_profit)
        data.loc[i,'Pip Value'] = get_pip_value(i)
        data.loc[i,'Position Size'] = get_position_size(i, stop_loss)
        data.loc[i, 'Bias'] = bias_results.get(i)
        data.loc[i,'Sentiment'] = get_sentiment(i)
        data.loc[i,'Signal'] = get_combined_signal(i)
    return data
        
data = get_data_table(pairs)

data


,Ask,Bid,price,EMA,RSI,ATR,Stop Loss,Take Profit,Round Stop Loss,Round Take Profit,Pip Value,Position Size,Bias,Sentiment,Signal
EURUSD,1.04057,1.04055,1.04055,1.046126,26.585422,0.001077,0.004309,0.008619,43.0,86.0,9.610302,2.01,neutral,sell,buy (0.5)
EURCHF,0.93569,0.93564,0.93564,0.938769,28.737271,0.000831,0.003325,0.006650,33.0,66.0,10.687529,2.36,sell,neutral,neutral
EURJPY,155.99700,155.98600,155.98600,156.428311,36.532162,0.247532,0.990127,1.980254,99.0,198.0,6.410585,1.31,neutral,neutral,buy (0.5)
USDCHF,0.89921,0.89918,0.89918,0.897379,56.788274,0.000858,0.003431,0.006862,34.0,69.0,11.120625,2.20,sell,buy,sell (0.5)
CHFJPY,166.72400,166.71100,166.71100,166.624836,51.565394,0.204403,0.817612,1.635225,82.0,164.0,5.998332,1.69,buy,neutral,neutral
USDJPY,149.91300,149.90700,149.90700,149.533595,58.753038,0.193221,0.772882,1.545764,77.0,155.0,6.669735,1.62,sell,neutral,neutral


# Grok Opinion

In [13]:
import time
import requests
import json
import re

xai_base_url = "https://api.x.ai/v1"
Grok_api = "xai-zBPdvxZOTUXY6Uaos1erYo0RIQultvO4E1TRvZ4eGhkNpV66ZYjH8Uke9Flf5iUp7xqg5SttSJyp9ccr"
xai_headers = {
    "Authorization": f"Bearer {Grok_api}",
    "Content-Type": "application/json"
}

def get_grok_signal(pair, data_row):
    """Get a probability-based trading signal from Grok-2-1212 API, always returning a float -1 to 1."""
    print(f"{pair} - Starting Grok signal fetch")
    try:
        pair_data = {
            "pair": pair,
            "price": float(data_row["price"].item()),  # Extract scalar
            "EMA": float(data_row["EMA"].item()),
            "RSI": float(data_row["RSI"].item()),
            "ATR": float(data_row["ATR"].item()),
            "bias": data_row["Bias"],  # String, no float needed
            "sentiment": data_row["Sentiment"],
            "signal": data_row["Signal"]
        }
        print(f"{pair} - Data prepared: {pair_data}")
        
        prompt = (
            "Analyze the forex pair data and return a single numerical score between -1 and 1 (e.g., '0.5' or '-0.3'), "
            "where -1 is a strong sell, 0 is neutral, and 1 is a strong buy, based on the likelihood of price movement "
            "given price, EMA, RSI, ATR, bias, sentiment, and ICT/RSI signals make your own decision and tell me what should i do. Return only the number, no text, no explanation.\n"
            f"Data: {json.dumps(pair_data, default=str)}"
        )
        
        payload = {
            "model": "grok-2-1212",
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 10
        }
        
        for attempt in range(3):
            try:
                print(f"{pair} - Attempt {attempt + 1}: Sending request")
                response = requests.post(f"{xai_base_url}/chat/completions", headers=xai_headers, json=payload, timeout=5)
                response.raise_for_status()
                grok_response = response.json()["choices"][0]["message"]["content"].strip()
                print(f"{pair} - Grok raw response: {grok_response}")
                
                match = re.search(r'-?\d*\.?\d+', grok_response)
                if match:
                    score = float(match.group())
                    if -1 <= score <= 1:
                        print(f"{pair} - Valid score: {score}")
                        return score
                    print(f"{pair} - Out of range score: {score}, retrying...")
                else:
                    print(f"{pair} - No number found: {grok_response}, retrying...")
            
            except requests.RequestException as e:
                error_msg = str(e)
                print(f"{pair} - Request error, attempt {attempt + 1}: {error_msg}")
                if "429" in error_msg:
                    print(f"{pair} - Rate limit hit, pausing...")
                    time.sleep(60)
                elif attempt < 2:
                    time.sleep(1)
            except (ValueError, KeyError) as e:
                print(f"{pair} - Parsing error, attempt {attempt + 1}: {e}")
                if attempt < 2:
                    time.sleep(1)
        
        print(f"{pair} - All attempts failed, falling back to RSI")
        return fallback_rsi_score(pair, data_row)
    
    except Exception as e:
        print(f"{pair} - Unexpected error before API call: {e}")
        return fallback_rsi_score(pair, data_row)

def fallback_rsi_score(pair, data_row):
    """Fallback to RSI-based score, always returning a float -1 to 1."""
    try:
        rsi = float(data_row["RSI"].item())  # Extract scalar
        score = 0.5 if rsi < 40 else -0.5 if rsi > 60 else 0
        print(f"{pair} - RSI fallback applied: {score} (RSI: {rsi})")
        return score
    except (KeyError, ValueError) as e:
        print(f"{pair} - Fallback error: {e}, defaulting to neutral")
        return 0.0

# Test code with throttling
data = get_data_table(pairs)
print(f"DataFrame index: {list(data.index)}")
for pair in pairs:
    print(f"Testing pair: {pair}")
    score = get_grok_signal(pair, data.loc[pair])
    print(f"{pair} - Grok Signal: {score}")
    time.sleep(5)

DataFrame index: ['EURUSD', 'EURCHF', 'EURJPY', 'USDCHF', 'CHFJPY', 'USDJPY']
Testing pair: EURUSD
EURUSD - Starting Grok signal fetch
EURUSD - Data prepared: {'pair': 'EURUSD', 'price': 1.04056, 'EMA': 1.0461259251599913, 'RSI': 26.612400590973294, 'ATR': 0.0010773412001785275, 'bias': 'neutral', 'sentiment': 'sell', 'signal': 'buy (0.5)'}
EURUSD - Attempt 1: Sending request
EURUSD - Grok raw response: -0.2
EURUSD - Valid score: -0.2
EURUSD - Grok Signal: -0.2
Testing pair: EURCHF
EURCHF - Starting Grok signal fetch
EURCHF - Data prepared: {'pair': 'EURCHF', 'price': 0.93574, 'EMA': 0.9387716608630786, 'RSI': 29.13298011322901, 'ATR': 0.0008312139922357529, 'bias': 'sell', 'sentiment': 'neutral', 'signal': 'neutral'}
EURCHF - Attempt 1: Sending request
EURCHF - Grok raw response: -0.6
EURCHF - Valid score: -0.6
EURCHF - Grok Signal: -0.6
Testing pair: EURJPY
EURJPY - Starting Grok signal fetch
EURJPY - Data prepared: {'pair': 'EURJPY', 'price': 156.011, 'EMA': 156.42908050124026, 'RSI

# Decision Taking Order Code

In [ ]:
import time
from datetime import datetime, timezone
import pandas as pd
import MetaTrader5 as mt5
from functools import lru_cache

MAX_DAILY_LOSS_PCT = 0.05
MAX_TOTAL_LOSS_PCT = 0.10

if not mt5.initialize():
    print("MT5 init failed.")
    exit()
signal_history = {pair: "neutral" for pair in pairs}
INITIAL_ACCOUNT_SIZE = None
LEVERAGE = None
realized_profit = 0

def initialize_account_params():
    global INITIAL_ACCOUNT_SIZE, LEVERAGE
    account_info = mt5.account_info()
    if not account_info:
        INITIAL_ACCOUNT_SIZE, LEVERAGE = 160000, 100
    else:
        INITIAL_ACCOUNT_SIZE, LEVERAGE = account_info.balance, account_info.leverage
    print(f"Init: ${INITIAL_ACCOUNT_SIZE}, {LEVERAGE}:1")

if INITIAL_ACCOUNT_SIZE is None:
    initialize_account_params()

def get_account_state():
    account_info = mt5.account_info()
    return (account_info.balance, account_info.equity) if account_info else (None, None)

def get_open_positions():
    positions = mt5.positions_get()
    open_trades = {}
    if positions:
        for pos in positions:
            open_trades[pos.symbol] = pos.type  # 0 = Buy, 1 = Sell
    return open_trades

def is_market_open(symbol):
    # Forex market is 24/5, assuming open Sunday 5 PM EST to Friday 5 PM EST
    utc_now = datetime.now(timezone.utc)
    weekday = utc_now.weekday()  # 0 = Monday, 6 = Sunday
    hour = utc_now.hour
    # Convert EST (UTC-5) to UTC: 5 PM EST = 22:00 UTC, adjust for your timezone if needed
    if weekday == 6 and hour >= 22:  # Sunday after 5 PM EST
        return True
    elif 0 <= weekday <= 4:  # Monday to Friday
        return True
    elif weekday == 5 and hour < 22:  # Friday before 5 PM EST
        return True
    return False

# Your original get_currency_data (confirmed working)
def get_currency_data(currency):
    dxy = yf.Ticker(currency)
    data = dxy.history(period="10d", interval="15m")
    data["RSI_16"] = ta.momentum.rsi(data["Close"], window=64)
    data["EMA_64"] = ta.trend.ema_indicator(data["Close"], window=196)
    latest_price = data['Close'].iloc[-1]
    latest_ema = data['EMA_64'].iloc[-1]
    latest_rsi = data['RSI_16'].iloc[-1]
    return latest_price, latest_ema, latest_rsi

# Cached wrapper to reduce API calls
@lru_cache(maxsize=4)
def get_currency_data_cached(currency, timestamp):
    try:
        return get_currency_data(currency)
    except Exception as e:
        print(f"Failed to fetch {currency}: {e}, using fallback")
        return (106.0, 106.0, 50.0)  # From your notebook output

def get_currencies_table(currencies):
    dict = {}
    current_time = int(time.time() // 300)  # Cache for 5 minutes
    for i in currencies:
        name = i
        if i == "DX-Y.NYB": name = "USD"
        elif i == "^XDE": name = "EUR"
        elif i == "^XDS": name = "CHF"
        elif i == "^XDN": name = "JPY"
        latest_price, latest_ema, latest_rsi = get_currency_data_cached(i, current_time)
        dict.update({name: [latest_price, latest_ema, latest_rsi]})
    return pd.DataFrame.from_dict(dict, orient='index', columns=['Price', 'EMA', 'RSI'])

def get_sentiment(pair):
    table = get_currencies_table(currencies)
    signal1 = "neutral"
    signal2 = "neutral"
    sentiment = "neutral"

    currency1, currency2 = None, None  # Ensure they are initialized

    for index in table.index:
        if pair.startswith(index):
            currency1 = index
        elif pair.endswith(index):
            currency2 = index

    # Ensure both currency1 and currency2 are found
    if currency1 is None or currency2 is None:
        raise ValueError(f"Could not determine currencies for pair: {pair}")

    if table.loc[currency1, "Price"] > table.loc[currency1, "EMA"] and table.loc[currency1, "RSI"] > 50:
        signal1 = "buy"
    elif table.loc[currency1, "Price"] < table.loc[currency1, "EMA"] and table.loc[currency1, "RSI"] < 50:
        signal1 = "sell"

    if table.loc[currency2, "Price"] > table.loc[currency2, "EMA"] and table.loc[currency2, "RSI"] > 50:
        signal2 = "buy"
    elif table.loc[currency2, "Price"] < table.loc[currency2, "EMA"] and table.loc[currency2, "RSI"] < 50:
        signal2 = "sell"

    if signal1 == "buy" and signal2 == "sell":
        sentiment = "buy"
    elif signal1 == "sell" and signal2 == "buy":
        sentiment = "sell"

    return sentiment


def get_data_table(pairs):
    dict = {}
    for i in pairs:
        ask, bid = mt5.symbol_info_tick(i).ask, mt5.symbol_info_tick(i).bid
        data = mt5.copy_rates_from_pos(i, mt5.TIMEFRAME_M15, 0, 64)
        df = pd.DataFrame(data)
        df['time'] = pd.to_datetime(df['time'], unit='s')
        df['EMA_64'] = ta.trend.ema_indicator(df['close'], window=64)
        df['RSI_16'] = ta.momentum.rsi(df['close'], window=16)
        df['ATR_16'] = ta.volatility.average_true_range(df['high'], df['low'], df['close'], window=16)
        latest_price, latest_ema, latest_rsi, latest_atr = df['close'].iloc[-1], df['EMA_64'].iloc[-1], df['RSI_16'].iloc[-1], df['ATR_16'].iloc[-1]
        dict.update({i: [ask, bid, latest_price, latest_ema, latest_rsi, latest_atr]})
    data = pd.DataFrame.from_dict(dict, orient='index', columns=['Ask', 'Bid', 'price', 'EMA', 'RSI', 'ATR'])
    data['Stop Loss'] = data['ATR'] * 4
    data['Take Profit'] = data['Stop Loss'] * 2
    for i in pairs:
        dec = 10000 if "JPY" not in i else 100
        data.loc[i, 'Round Stop Loss'] = int(round(data.loc[i, "Stop Loss"] * dec))
        data.loc[i, 'Round Take Profit'] = int(round(data.loc[i, "Take Profit"] * dec))
        data.loc[i, 'Sentiment'] = get_sentiment(i)
        data.loc[i, 'Bias'] = "neutral"  # Simplified, use your bias_for_pairs if needed
        data.loc[i, 'Signal'] = get_combined_signal(i)
    return data

def get_combined_signal(symbol):
    # Simplified from your original, adjust as needed
    data = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_M15, 0, 64)
    if data is None:
        return "neutral"
    df = pd.DataFrame(data)
    df['RSI_16'] = ta.momentum.rsi(df['close'], window=16)
    rsi = df['RSI_16'].iloc[-1]
    return "buy (0.5)" if rsi < 40 else "sell (0.5)" if rsi > 60 else "neutral"

def get_ftmo_position_size(pair, stop_loss_pips):
    balance, equity = get_account_state()
    if not balance:
        return 0.0
    risk_amount = balance * 0.005  # 0.5% risk
    pip_value = (0.0001 * 100000 / data.loc[pair, 'price']) if "JPY" not in pair else (0.01 * 100000 / data.loc[pair, 'price'])
    return round(risk_amount / (stop_loss_pips * pip_value), 2)

def calculate_score(bias, sentiment, combined_signal, pair, data_row):
    bias_score = 1 if bias == "buy" else -1 if bias == "sell" else 0
    sentiment_score = 1 if sentiment == "buy" else -1 if sentiment == "sell" else 0
    signal_score = 1 if "buy" in combined_signal else -1 if "sell" in combined_signal else 0
    signal_strength = float(combined_signal.split('(')[1].strip(')')) if '(' in combined_signal else 0
    return (0.3 * bias_score) + (0.3 * sentiment_score) + (0.4 * signal_score * signal_strength)

def send_order(symbol, action, volume, stop_loss, take_profit):
    open_trades = get_open_positions()
    if symbol in open_trades:
        existing_trade = open_trades[symbol]
        if (existing_trade == 0 and action in ['buy', 'strong buy']) or (existing_trade == 1 and action in ['sell', 'strong sell']):
            return False, f"{symbol} trade skipped: Position already open."
        elif (existing_trade == 0 and action in ['sell', 'strong sell']) or (existing_trade == 1 and action in ['buy', 'strong buy']):
            return False, f"{symbol} trade skipped: Contradictory trade detected."
    
    if not mt5.initialize() or not is_market_open(symbol):
        return False, f"{symbol} trade failed: MT5 or market issue"
    symbol_info = mt5.symbol_info(symbol)
    if not symbol_info:
        return False, f"{symbol} info failed"
    
    price = mt5.symbol_info_tick(symbol).ask if action in ['buy', 'strong buy'] else mt5.symbol_info_tick(symbol).bid
    sl_price = price - stop_loss if action in ['buy', 'strong buy'] else price + stop_loss
    tp_price = price + take_profit if action in ['buy', 'strong buy'] else price - take_profit
    
    order_type = mt5.ORDER_TYPE_BUY if action in ['buy', 'strong buy'] else mt5.ORDER_TYPE_SELL
    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": symbol,
        "volume": round(volume, 2),
        "type": order_type,
        "price": price,
        "sl": sl_price,
        "tp": tp_price,
        "deviation": 10,
        "magic": 234000,
        "comment": "Auto",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC
    }
    
    result = mt5.order_send(request)
    if result.retcode != mt5.TRADE_RETCODE_DONE:
        return False, f"{symbol} order failed: {result.comment}"
    global realized_profit
    realized_profit += (tp_price - price) * volume * 100000 if action in ['buy', 'strong buy'] else (price - tp_price) * volume * 100000
    return True, f"{symbol} {action} {volume:.2f} lots placed."

last_summary_time = None
actions_taken = []

while True:
    try:
        current_time = datetime.now(timezone.utc)
        if not any(is_market_open(pair) for pair in pairs):
            print(f"Market closed at {current_time}, waiting...")
            time.sleep(900)  # 15 minutes during off-hours
            continue
        
        data = get_data_table(pairs)
        market_open = False
        open_trades = get_open_positions()
    
        for pair in pairs:
            bias = data.loc[pair, 'Bias']
            sentiment = data.loc[pair, 'Sentiment']
            combined_signal = data.loc[pair, 'Signal']
            stop_loss_pips = data.loc[pair, 'Round Stop Loss']
            stop_loss = stop_loss_pips / (10000 if "JPY" not in pair else 100)
            take_profit = data.loc[pair, 'Round Take Profit'] / (10000 if "JPY" not in pair else 100)
            
            if is_market_open(pair):
                market_open = True
                last_signal = signal_history[pair]
                if combined_signal != last_signal and "neutral" not in combined_signal:
                    score = calculate_score(bias, sentiment, combined_signal, pair, data.loc[pair])
                    if abs(score) > 0 :
                        base_size = get_ftmo_position_size(pair, stop_loss_pips)  
                        if base_size > 0:
                            if score <= -0.501:
                                action = 'strong sell'
                                adjusted_size = base_size + (base_size *abs(score))
                            elif -0.501 <= score < -0.01:
                                action = 'sell'
                                adjusted_size = base_size - (base_size *abs(score))
                            elif 0.01 <= score < 0.501:
                                action = 'buy'
                                adjusted_size = base_size - (base_size *abs(score))
                            elif score >= 0.501:
                                action = 'strong buy'
                                adjusted_size = base_size + (base_size *abs(score))
                            else:
                                continue
                            
                            success, message = send_order(pair, action, adjusted_size, stop_loss, take_profit)
                            if success:
                                actions_taken.append(f"{message} Score: {score:.2f}")
                                signal_history[pair] = combined_signal
            time.sleep(5)

        if last_summary_time is None or (current_time - last_summary_time).total_seconds() >= 1800:
            balance, equity = get_account_state()
            balance = balance or INITIAL_ACCOUNT_SIZE
            print(f"Summary at {current_time}: {'Closed' if not market_open else 'Open'}")
            if actions_taken:
                for action in actions_taken:
                    print(f"  {action}")
            else:
                print("  No trades")
            print(f"Balance: ${balance:.2f}, Equity: ${equity:.2f}, Profit: ${realized_profit:.2f}")
            actions_taken = []
            last_summary_time = current_time
    
    except Exception as e:
        print(f"Error: {e}")
    
    time.sleep(60)  # 1-minute loop

Init: $166179.81, 100:1
Summary at 2025-02-27 16:10:05.840797+00:00: Open
  EURUSD sell 1.81 lots placed. Score: -0.10
  EURCHF buy 1.89 lots placed. Score: 0.20
  USDJPY sell 1.30 lots placed. Score: -0.20
Balance: $166172.61, Equity: $165252.54, Profit: $203681.82
